In [1]:
import os
import numpy as np
import onnx
import onnxruntime as ort

from transformers import AutoTokenizer

import aimet_onnx
from aimet_onnx.common.defs import QuantScheme
from aimet_onnx import QuantizationSimModel

#FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model_simplified.onnx"
#FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_fix_mask.onnx"
#FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model.onnx"  # ← 適宜変更
FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_mask_add.onnx"
OUT_DIR = "/root/AIMET_ENV/tools/ruri/model"

os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "/root/AIMET_ENV/tools/ruri/ruri-small-v2"
# tokenizer は事前にロード
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# 1) load onnx
model = onnx.load_model(FP32_ONNX)

# 2) (推奨) simplify
"""
try:
    import onnxsim
    model, _ = onnxsim.simplify(model)
except Exception as e:
    print("onnxsim simplify failed, continue with original model:", repr(e))
"""
import onnx



model = onnx.load_model(FP32_ONNX)

def fix_input_shapes(model, shapes_dict):
    for inp in model.graph.input:
        name = inp.name
        if name in shapes_dict:
            shape = shapes_dict[name]
            for i, dim in enumerate(inp.type.tensor_type.shape.dim):
                dim.dim_value = shape[i]

    return model


si_model = fix_input_shapes(
    model,
    {
        "input_ids":      [1, 512],
        "attention_mask": [1, 512],
        "token_type_ids": [1, 512],
        "position_ids":   [1, 512],
    }
)


# 3) create QuantSim (CPU only, W8/A16)
providers = ["CPUExecutionProvider"]



sim = QuantizationSimModel(

    model,
    param_type=aimet_onnx.int8,
    activation_type=aimet_onnx.int16,
    #quant_scheme=QuantScheme.min_max,
    quant_scheme=QuantScheme.post_training_tf_enhanced,
    #config_file="default",
    config_file="htp_v73",
    #config_file="/root/AIMET_ruri_v2/tool/quantsim_fp_escape.json",  # ← 適宜変更
    providers=providers,
)



/root/AIMET_ENV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-19 07:36:28,542 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:V73


In [2]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


class AimetOnnxEmbedder:
    def __init__(self, sim, tokenizer, session=None, max_length=512, pool="mean"):
        self.sim = sim
        self.tokenizer = tokenizer
        self.session = session
        self.max_length = max_length
        self.pool = pool

    def _pool(self, last_hidden_state, attention_mask):
        if self.pool == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.astype(np.float32)
        denom = np.clip(mask.sum(axis=1, keepdims=True), 1e-6, None)
        return (last_hidden_state * mask[:, :, None]).sum(axis=1) / denom

    def encode(self, sentences, batch_size=1, **kwargs):
        embs = []
        sess = self.session or self.sim.session
        required = {i.name for i in sess.get_inputs()}

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            tok = self.tokenizer(
                batch,
                padding="max_length",   # ★固定512に必須
                truncation=True,
                max_length=512,         # ★Expected: 512 に合わせる
                return_tensors="np",
            )

            input_ids = tok["input_ids"].astype(np.int64)
            attention_mask = tok["attention_mask"].astype(np.int64)

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }

            # token_type_ids（無いなら0埋め）
            if "token_type_ids" in required:
                if "token_type_ids" in tok:
                    inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
                else:
                    inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)

            # position_ids（無いなら作る）
            if "position_ids" in required:
                bsz, seqlen = input_ids.shape  # seqlen は 512 のはず
                pos = np.arange(seqlen, dtype=np.int64)[None, :]
                inputs["position_ids"] = np.repeat(pos, bsz, axis=0)

            outputs = sess.run(None, inputs)

            last_hidden_state = outputs[0]
            """
            pooled = self._pool(last_hidden_state, attention_mask).astype(np.float32)
            pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
            embs.append(pooled)
            """
            pooled = outputs[1].astype(np.float32)
        
            embs.append(pooled)
        return np.vstack(embs)


class TextEmbedderCompat:
    def __init__(self, embedder: AimetOnnxEmbedder, batch_size=1):
        self.embedder = embedder
        self.batch_size = batch_size

    def set_output_tensor(self):  # called by RetrievalEvaluator
        return

    def reset_max_seq_length(self):  # only used if force_max_length=True
        return

    def batch_encode_with_cache(self, text_list, prefix=None, cache_path=None, overwrite_cache=False, **kwargs):
        if prefix:
            text_list = [prefix + t for t in text_list]
        return self.embedder.encode(text_list, batch_size=self.batch_size)

import json
import dataclasses
from pathlib import Path

def _to_jsonable(obj):
    # dataclass -> dict
    if dataclasses.is_dataclass(obj):
        return {k: _to_jsonable(v) for k, v in dataclasses.asdict(obj).items()}
    # numpy -> python
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    # dict/list
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(x) for x in obj]
    # その他はそのまま（JSONが無理ならstrにする）
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return str(obj)

def save_results(results, out_path: str):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    payload = _to_jsonable(results)
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", out_path)


In [ ]:

def main():
    

    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="nlp_journal_abs_intro-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    #model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)
    import onnxruntime as ort
    ort_session = ort.InferenceSession(si_model.SerializePartialToString(), providers=providers)
    
    embedder = AimetOnnxEmbedder(
        sim=None,
        tokenizer=tokenizer,
        session=ort_session   # ←これ重要
    )

    model = TextEmbedderCompat(embedder, batch_size=1)


    results = evaluator(model, cache_dir="cache_abs_intro", overwrite_cache=False)
    print(results)
    save_results(results, "results/abs_intro_normal.json")
if __name__ == "__main__":
    main()

2026-05-14 07:46:47.025 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_intro-corpus) with split corpus


Fail: [ONNXRuntimeError] : 1 : FAIL : Fatal error: aimet.customop.cpu:QcQuantizeOp(-1) is not a registered function/op

2026-05-14 07:49:02.513 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_intro-corpus) with split corpus


IndexError: list index out of range

In [ ]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


def main():
    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="nlp_journal_title_abs-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_title_abs-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)

    results = evaluator(model, cache_dir="cache_title_abs", overwrite_cache=False)
    print(results)
    save_results(results, "results/title_abs_mask_calib_pad_jmpad.json")
if __name__ == "__main__":
    main()

2026-05-14 07:32:47.894 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_title_abs-corpus) with split corpus


In [13]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


def main():
    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="nlp_journal_abs_article-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_article-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)

    results = evaluator(model, cache_dir="cache_title_abs", overwrite_cache=False)
    print(results)
    save_results(results, "results/title_abs_normal.json")
if __name__ == "__main__":
    main()

2026-05-13 06:52:31.272 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_article-corpus) with split corpus
2026-05-13 06:54:52.734 | INFO     | jmteb.evaluators.retrieval.evaluator:__call__:165 - Start retrieval
Retrieval doc chunks: 100%|██████████| 637/637 [00:00<00:00, 13528.16it/s]


EvaluationResults(metric_name='ndcg@10', metric_value=0.9032982265408709, details={'optimal_distance_metric': 'cosine_similarity', 'val_scores': {'cosine_similarity': {'accuracy@1': 0.8333333333333334, 'accuracy@3': 0.9215686274509803, 'accuracy@5': 0.9529411764705882, 'accuracy@10': 0.9666666666666667, 'accuracy@20': 0.9725490196078431, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9032982265408709), 'mrr@50': 0.8832860827614456}, 'dot_score': {'accuracy@1': 0.8333333333333334, 'accuracy@3': 0.9215686274509803, 'accuracy@5': 0.9529411764705882, 'accuracy@10': 0.9666666666666667, 'accuracy@20': 0.9725490196078431, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9032982265408709), 'mrr@50': 0.8832860827614456}, 'euclidean_distance': {'accuracy@1': 0.8333333333333334, 'accuracy@3': 0.9215686274509803, 'accuracy@5': 0.9529411764705882, 'accuracy@10': 0.9666666666666667, 'accuracy@20': 0.97254901960

In [ ]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


def main():
    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="nlp_journal_title_intro-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_title_intro-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)

    results = evaluator(model, cache_dir="cache_title_intro", overwrite_cache=False)
    print(results)
    save_results(results, "results/title_intro_normal.json")
if __name__ == "__main__":
    main()

In [15]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


def main():
    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="jagovfaqs_22k-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="jagovfaqs_22k-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)

    results = evaluator(model, cache_dir="cache_jagov", overwrite_cache=False)
    print(results)
    save_results(results, "results/jagov_normal.json")
if __name__ == "__main__":
    main()

2026-05-12 06:23:34.085 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=jagovfaqs_22k-corpus) with split corpus
2026-05-12 07:06:13.302 | INFO     | jmteb.evaluators.retrieval.evaluator:__call__:165 - Start retrieval
Retrieval doc chunks: 100%|██████████| 22794/22794 [00:00<00:00, 268844.02it/s]


EvaluationResults(metric_name='ndcg@10', metric_value=0.016483852224009773, details={'optimal_distance_metric': 'cosine_similarity', 'val_scores': {'cosine_similarity': {'accuracy@1': 0.01023391812865497, 'accuracy@3': 0.014912280701754385, 'accuracy@5': 0.0195906432748538, 'accuracy@10': 0.024269005847953218, 'accuracy@20': 0.030116959064327486, 'accuracy@30': 0.031871345029239766, 'accuracy@50': 0.03508771929824561, 'ndcg@10': np.float64(0.016483852224009773), 'mrr@50': 0.014630231328992573}, 'dot_score': {'accuracy@1': 0.009941520467836258, 'accuracy@3': 0.014912280701754385, 'accuracy@5': 0.0195906432748538, 'accuracy@10': 0.024269005847953218, 'accuracy@20': 0.030116959064327486, 'accuracy@30': 0.031871345029239766, 'accuracy@50': 0.03508771929824561, 'ndcg@10': np.float64(0.016375936947276282), 'mrr@50': 0.014483177534662692}, 'euclidean_distance': {'accuracy@1': 0.009941520467836258, 'accuracy@3': 0.014912280701754385, 'accuracy@5': 0.0195906432748538, 'accuracy@10': 0.024269005

In [6]:
import numpy as np

def get_embedding(text: str, sim=sim, tokenizer=tokenizer, max_length=512, pool="mean"):
    """
    テキストをsimモデルに渡してembeddingベクトルを返す
    """
    required = {i.name for i in sim.session.get_inputs()}
    
    tok = tokenizer(
        [text],
        #padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="np",
    )
    
    input_ids = tok["input_ids"].astype(np.int64)
    attention_mask = tok["attention_mask"].astype(np.int64)
    
    inputs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }
    
    if "token_type_ids" in required:
        if "token_type_ids" in tok:
            inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
        else:
            inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)
    
    if "position_ids" in required:
        bsz, seqlen = input_ids.shape
        inputs["position_ids"] = np.tile(np.arange(seqlen, dtype=np.int64), (bsz, 1))
    
    # 推論実行
    outputs = sim.session.run(None, inputs)
    last_hidden_state = outputs[0]  # shape: (1, seq_len, hidden_size)
    print("last_hidden_state shape:", last_hidden_state.shape)
    last_hidden_state.astype(np.float32).tofile("/root/AIMET_ENV/tools/comp_raw/last_hidden_state_normal_nopad.raw")  # デバッグ用
    # pooling
    
    # output.raw 相当: shape (hidden_size,) の numpy array
    output_raw = outputs[1]
    return output_raw


# 使用例
text = "クエリ: 瑠璃色はどんな色？"
output_raw = get_embedding(text)
print("output.raw shape:", output_raw.shape)
#print("output.raw:", output_raw[:10], "...")  # 先頭10次元を表示
output_raw.astype(np.float32).tofile("/root/AIMET_ENV/tools/comp_raw/pooled_output_normal_nopad.raw")

last_hidden_state shape: (1, 14, 768)
output.raw shape: (1, 768)


In [27]:
# 2つのRAWファイルを読み込んで比較
raw_path1 = "/root/AIMET_ENV/tools/ruri/convert/sim_output.raw"
raw_path2 = "/root/AIMET_ENV/tools/ruri/convert/sim_output2.raw"

vec1 = np.fromfile(raw_path1, dtype=np.float32)
vec2 = np.fromfile(raw_path2, dtype=np.float32)

# コサイン類似度計算
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

sim_score = cosine_similarity(vec1, vec2)
print(f"類似度: {sim_score:.4f}")


vec1 shape: (768,)

基準テキスト: 'クエリ: 瑠璃色はどんな色？'
------------------------------------------------------------
類似度: 1.0000  |  'クエリ: 瑠璃色はどんな色？'
類似度: 0.9497  |  '文章: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズ...' 
類似度: 0.8172  |  'クエリ: ワシやタカのように、鋭いくちばしと爪を持った大型の鳥類を総称して「何類...' 
類似度: 0.8128  |  '文章: 全く関係のない文章です。今日の天気は晴れです。'


In [15]:
# 既存の FP32_ONNX を使ってローカル ONNX を読み込み、input / output の shape を表示
import onnx

graph = model.graph
model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)
def _shape_from_value_info(value_info):
    t = value_info.type
    if not t.HasField("tensor_type"):
        return "non-tensor"
    dims = []
    for d in t.tensor_type.shape.dim:
        if d.HasField("dim_value"):
            dims.append(d.dim_value)
        elif d.HasField("dim_param"):
            dims.append(d.dim_param)  # 例: "batch", "sequence"
        else:
            dims.append("?")
    return dims

print(f"ONNX: {FP32_ONNX}")

print("\n[Inputs]")
for vi in graph.input:
    print(f"- {vi.name}: {_shape_from_value_info(vi)}")

print("\n[Outputs]")
for vi in graph.output:
    print(f"- {vi.name}: {_shape_from_value_info(vi)}")

ONNX: /root/AIMET_ruri/ruri-onnx/model.onnx

[Inputs]
- input_ids: [1, 512]
- attention_mask: [1, 512]

[Outputs]
- last_hidden_state_updated: [1, 512, 768]


In [ ]:
# サンプル1件
text = calib_texts[11]
enc = tokenizer(
    text,
    #padding="max_length",
    truncation=True,
    max_length=512,
    return_tensors="np",
)

input_ids = enc["input_ids"][0]
attention_mask = enc["attention_mask"][0]

# 長さ確認
print("input_ids length:", len(input_ids))
print("attention_mask length:", len(attention_mask))

# 有効トークン数（=1の数）
valid_tokens = attention_mask.sum()
print("valid_tokens:", valid_tokens)

# 後ろ50トークン確認（PAD確認）
print("\n--- tail 50 tokens ---")
print("input_ids:", input_ids[-50:])
print("attention_mask:", attention_mask[-50:])

In [31]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


class AimetOnnxEmbedder:
    def __init__(self, sim, tokenizer, session=None, max_length=512, pool="mean"):
        self.sim = sim
        self.tokenizer = tokenizer
        self.session = session
        self.max_length = max_length
        self.pool = pool

    def _pool(self, last_hidden_state, attention_mask):
        if self.pool == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.astype(np.float32)
        denom = np.clip(mask.sum(axis=1, keepdims=True), 1e-6, None)
        return (last_hidden_state * mask[:, :, None]).sum(axis=1) / denom

    def encode(self, sentences, batch_size=1, **kwargs):
        embs = []
        sess = self.session or self.sim.session
        required = {i.name for i in sess.get_inputs()}

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            tok = self.tokenizer(
                batch,
                padding="max_length",   # ★固定512に必須
                truncation=True,
                max_length=512,         # ★Expected: 512 に合わせる
                return_tensors="np",
            )

            input_ids = tok["input_ids"].astype(np.int64)
            attention_mask = tok["attention_mask"].astype(np.int64)

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }

            # token_type_ids（無いなら0埋め）
            if "token_type_ids" in required:
                if "token_type_ids" in tok:
                    inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
                else:
                    inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)

            # position_ids（無いなら作る）
            if "position_ids" in required:
                bsz, seqlen = input_ids.shape  # seqlen は 512 のはず
                pos = np.arange(seqlen, dtype=np.int64)[None, :]
                inputs["position_ids"] = np.repeat(pos, bsz, axis=0)

            outputs = sess.run(None, inputs)

            last_hidden_state = outputs[0]
            
            #pooled = self._pool(last_hidden_state, attention_mask).astype(np.float32)
            #pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
            #embs.append(pooled)
            
            pooled = outputs[1].astype(np.float32)
        
            # normalizeだけかける（もし必要なら）
            pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
            embs.append(pooled)
        return np.vstack(embs)


class TextEmbedderCompat:
    def __init__(self, embedder: AimetOnnxEmbedder, batch_size=1):
        self.embedder = embedder
        self.batch_size = batch_size

    def set_output_tensor(self):  # called by RetrievalEvaluator
        return

    def reset_max_seq_length(self):  # only used if force_max_length=True
        return

    def batch_encode_with_cache(self, text_list, prefix=None, cache_path=None, overwrite_cache=False, **kwargs):
        if prefix:
            text_list = [prefix + t for t in text_list]
        return self.embedder.encode(text_list, batch_size=self.batch_size)

import json
import dataclasses
from pathlib import Path

def _to_jsonable(obj):
    # dataclass -> dict
    if dataclasses.is_dataclass(obj):
        return {k: _to_jsonable(v) for k, v in dataclasses.asdict(obj).items()}
    # numpy -> python
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    # dict/list
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(x) for x in obj]
    # その他はそのまま（JSONが無理ならstrにする）
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return str(obj)

def save_results(results, out_path: str):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    payload = _to_jsonable(results)
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", out_path)


In [32]:
import os
import onnx
import onnxruntime as ort
from transformers import AutoTokenizer

FP32_ONNX = "/root/AIMET_ENV/tools/ruri/quantize/ruri-export-onnx/model_mask_add.onnx"
MODEL_ID = "/root/AIMET_ENV/tools/ruri/ruri-small-v2"

providers = ["CPUExecutionProvider"]

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

def fix_input_shapes(model, shapes_dict):
    for inp in model.graph.input:
        name = inp.name
        if name in shapes_dict:
            shape = shapes_dict[name]
            for i, dim in enumerate(inp.type.tensor_type.shape.dim):
                dim.dim_value = shape[i]
    return model


def build_fp32_session():
    model = onnx.load_model(FP32_ONNX)

    model = fix_input_shapes(
        model,
        {
            "input_ids": [1, 512],
            "attention_mask": [1, 512],
            "token_type_ids": [1, 512],
            "position_ids": [1, 512],
        }
    )

    session = ort.InferenceSession(
        model.SerializeToString(),
        providers=providers,
    )
    return session


def main():
    DATASET_ROOT = "/root/jmteb_260326"

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    ort_session = build_fp32_session()

    embedder = AimetOnnxEmbedder(
        sim=None,
        tokenizer=tokenizer,
        session=ort_session,
    )

    model = TextEmbedderCompat(embedder, batch_size=1)

    results = evaluator(model, cache_dir="cache_abs_intro", overwrite_cache=False)
    print(results)
    save_results(results, "results/abs_intro_fp32.json")


if __name__ == "__main__":
    main()

2026-05-14 08:58:24.047 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_intro-corpus) with split corpus
2026-05-14 08:59:12.316 | INFO     | jmteb.evaluators.retrieval.evaluator:__call__:165 - Start retrieval
Retrieval doc chunks: 100%|██████████| 637/637 [00:00<00:00, 13380.60it/s]


EvaluationResults(metric_name='ndcg@10', metric_value=0.9044389931055391, details={'optimal_distance_metric': 'cosine_similarity', 'val_scores': {'cosine_similarity': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.9705882352941176, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9044389931055391), 'mrr@50': 0.8881360771421324}, 'dot_score': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.9705882352941176, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9044389931055391), 'mrr@50': 0.8881360771421324}, 'euclidean_distance': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.97058823529